# Who Gets Paid More? The Socio-Demographic Influences of Income
**Data Exploration and Preparation (DEP) - CCT College Dublin**  
**Author:** Rhuan Mendanha | **Topic:** Population and Society  
**Dataset:** UCI Adult / Census Income (Becker and Kohavi, 1996), The US Census of 1994 - 48,842 rows and 15 columns

### Research Questions
- **Q1.** How does the share of high earners (>50K) differ across the various social categories?
- **Q2.** Does education have the equal pay off for both genders?
- **Q3.** What is the relationship between age, work hours per week and income level?
- **Q4.** What attributes show the most correlation with income, and what unusual phenomena exist?

### Structure
1. Loading, Documentation and Cleaning of Data
2. Exploratory Data Analysis (variation, covariance, unusual phenomena)
3. Encoding (labels, ordinal, one-hot, PCA)

**Running Instructions:** keep `adult-all.csv` in the same folder as this notebook and select menu item *Run > Run All Cells*.

## Setup - libraries and chart style

In [1]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from scipy.stats import chi2_contingency
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler

%matplotlib inline

In [2]:
# ---------- Setting up the graphs ----------
# I wanted all my graphs to have the same style so the project looks more organised.
# I chose colours that are easy to distinguish and kept the design simple
# so the information is easier to understand.

SAVE_FIGURES = False          # I only turn this on when I need to save graphs for my poster
FIG = Path("figures")        # This is the folder where I will save my graphs


class ps:
    # These are the colours I use throughout my graphs to keep everything consistent.
    BLUE = "#2a78d6"     # Used for the first group, such as <=50K or Female
    ORANGE = "#eb6834"   # Used for the second group, such as >50K or Male
    AQUA = "#1baf7a"
    INK = "#0b0b0b"
    INK2 = "#52514e"
    GRID = "#e4e3df"

    @staticmethod
    def apply():
        # Here I set up the general appearance of my graphs.
        # I adjusted the fonts, colours, grid lines and borders to make
        # the graphs look cleaner and easier to read.
        mpl.rcParams.update({
            "figure.dpi": 110, "savefig.dpi": 220, "savefig.bbox": "tight",
            "font.family": "DejaVu Sans", "font.size": 14,
            "axes.titlesize": 16, "axes.titleweight": "bold",
            "axes.titlelocation": "left", "axes.labelcolor": ps.INK2,
            "axes.edgecolor": ps.GRID, "axes.spines.top": False,
            "axes.spines.right": False, "axes.grid": True,
            "grid.color": ps.GRID, "grid.linewidth": 0.8,
            "xtick.color": ps.INK2, "ytick.color": ps.INK2,
            "text.color": ps.INK, "legend.frameon": False,
            "lines.linewidth": 2.2,
        })

    @staticmethod
    def save(fig, path):
        # I use this function whenever I finish a graph.
        # It displays the graph in my notebook and saves a copy if I need it later.
        if SAVE_FIGURES:
            FIG.mkdir(exist_ok=True)
            fig.savefig(path, facecolor="white")
        plt.show()


# I apply my graph settings here so all the graphs I create afterwards
# automatically follow the same style.
ps.apply()

## Part 1 - Data loading, characterisation and cleaning

Inferences from the dataset: UCI Adult / Census Income (Becker & Kohavi, 1996). The dataset consists of 48,842 rows and 15 columns and has been extracted from 1994 US Census Bureau database. 

Procedure:
1. Reading the raw CSV file (akin to removing the headers of the original file) and renaming the columns.
2. Description of the data set with respect to its size, features and data types.
3. Providing the solution for the missing values present in the data set (missing values defined as “?” in the original data).
4. Eliminating the duplicate records from the data file.
5. Identifying the outlier records by applying IQR rule and z-score.
6. Developing some features to use them in the Exploratory Data Analysis phase.
7. Saving the processed data set into the file – “adult_clean.csv” and the cleaning report in “cleaning_report.txt”.

### Paths

In [3]:
RAW = Path("adult-all.csv")   # This is the original dataset file, which I keep in the same folder as my notebook.

# The original dataset doesn't have column names, so I added them myself
# using the names provided by UCI. This makes it easier to work with the data
# and understand what each column represents.
COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country",
    "income",
]

# I use this list to keep track of everything I do during the data cleaning process.
# This way, I can go back later and see what changes were made to the dataset.
report = []


def log(msg: str = "") -> None:
    # This function prints each message on screen and also saves it to my report.
    # This helps me keep a record of the steps I take while cleaning the data.
    print(msg)
    report.append(str(msg))

### 1. Load

In [4]:
# I load the original CSV file into a DataFrame and add the column names I defined earlier.
# I also remove extra spaces after commas so the data is easier to work with.
df = pd.read_csv(RAW, header=None, names=COLUMNS, skipinitialspace=True)


# I noticed that some text values can have extra spaces around them.
# I remove those spaces to avoid treating the same value as two different categories.
obj_cols = df.select_dtypes(exclude="number").columns
df[obj_cols] = df[obj_cols].apply(lambda s: s.str.strip())


# Some versions of the dataset have a full stop after the income values (e.g. '>50K.').
# I remove it so that all income labels follow the same format.
df["income"] = df["income"].str.replace(".", "", regex=False)

### 2. Characterisation

In [5]:
# ---------- 1. Understanding the original dataset ----------
# Before making any changes, I wanted to get a better idea of what my dataset looks like.
# Here I check its size, identify the numerical and categorical columns,
# and see how many different values each column contains.

log("=== 1. RAW DATA CHARACTERISATION ===")

# Shows how many rows and columns the dataset has.
log(f"Shape (rows, columns): {df.shape}")

# Separates the columns containing numbers from the categorical ones.
log(f"Numeric attributes    : {list(df.select_dtypes('number').columns)}")
log(f"Categorical attributes: {list(df.select_dtypes(exclude='number').columns)}")

# Checks how many unique values each column contains.
# This helps me understand the variety of information in the dataset.
log("Unique values per column:")
log(df.nunique().to_string())

# Adds an empty line to make the cleaning report easier to read.
log("")

=== 1. RAW DATA CHARACTERISATION ===
Shape (rows, columns): (48842, 15)
Numeric attributes    : ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
Categorical attributes: ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country', 'income']
Unique values per column:
age                  74
workclass             9
fnlwgt            28523
education            16
education_num        16
marital_status        7
occupation           15
relationship          6
race                  5
sex                   2
capital_gain        123
capital_loss         99
hours_per_week       96
native_country       42
income                2



### 3. Missing values


In [6]:
# ---------- 2. Checking and handling missing values ----------
# Some missing values in the original dataset are written as '?' instead of being empty.
# I replace them with NaN so pandas can recognise them properly and count them.

df = df.replace("?", np.nan)

# I check which columns have missing values and calculate how many there are.
# I also calculate the percentage of missing data in each column.
missing = df.isna().sum()
missing = missing[missing > 0]

log("=== 2. MISSING VALUES (before treatment) ===")
log(pd.DataFrame({"n_missing": missing,
                  "pct": (missing / len(df) * 100).round(2)}).to_string())

# Checks how many rows have at least one missing value.
log(f"Rows with at least one missing value: {df.isna().any(axis=1).sum()}")


# I wanted to check whether missing occupation values could affect my results.
# To do this, I compare the percentage of people earning over 50K
# between those with and without a recorded occupation.
occ_missing = df["occupation"].isna()
rate_missing = (df.loc[occ_missing, "income"] == ">50K").mean()
rate_present = (df.loc[~occ_missing, "income"] == ">50K").mean()

log(f">50K rate when occupation missing : {rate_missing:.3f}")
log(f">50K rate when occupation present : {rate_present:.3f}")

# I also check how many rows are missing both workclass and occupation.
both = (df["workclass"].isna() & df["occupation"].isna()).sum()
log(f"Rows where workclass AND occupation are both missing: {both}")
log("")


# ---------- Handling the missing values ----------
# Instead of deleting rows with missing information, I decided to keep them.
# Workclass and occupation are often missing together, so I label these values
# as 'Unknown' to avoid losing potentially useful information.
#
# Native country has very few missing values, so I replace them with the most
# common country in the dataset (the mode).

df["workclass"] = df["workclass"].fillna("Unknown")
df["occupation"] = df["occupation"].fillna("Unknown")

mode_country = df["native_country"].mode()[0]
df["native_country"] = df["native_country"].fillna(mode_country)

# I record the changes in my report and check if anything is still missing.
log(f"workclass/occupation NaN -> 'Unknown'; native_country NaN -> mode ('{mode_country}')")
log(f"Missing values after treatment: {int(df.isna().sum().sum())}")
log("")

=== 2. MISSING VALUES (before treatment) ===
                n_missing   pct
workclass            2799  5.73
occupation           2809  5.75
native_country        857  1.75
Rows with at least one missing value: 3620
>50K rate when occupation missing : 0.094
>50K rate when occupation present : 0.248
Rows where workclass AND occupation are both missing: 2799

workclass/occupation NaN -> 'Unknown'; native_country NaN -> mode ('United-States')
Missing values after treatment: 0



### 4. Duplicates

In [7]:
# ---------- 3. Checking and removing duplicate rows ----------
# I check if the dataset contains any completely duplicated rows.
# Duplicate records could affect my analysis, so I remove them to keep
# the dataset as accurate and consistent as possible.

n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

# I record how many duplicates were removed and check the new dataset size.
log("=== 3. DUPLICATES ===")
log(f"Exact duplicate rows removed: {n_dup}  ->  new shape {df.shape}")
log("")

=== 3. DUPLICATES ===
Exact duplicate rows removed: 52  ->  new shape (48790, 15)



### 4b. Logical inconsistencies

In [8]:
# ---------- 3b. Checking inconsistent records ----------
# I wanted to check if there were any records where the relationship and sex
# information did not match. For example, someone listed as a Husband but
# recorded as Female, or a Wife recorded as Male.
# I treat these cases as possible data-entry errors and remove them.

bad = ((df["relationship"] == "Husband") & (df["sex"] == "Female")) | \
      ((df["relationship"] == "Wife") & (df["sex"] == "Male"))

# I count the inconsistent records before removing them.
log("=== 3b. INCONSISTENT RECORDS ===")
log(f"Husband/Female or Wife/Male contradictions removed: {bad.sum()}")

# Removes the records that don't match and resets the row numbering.
df = df.loc[~bad].reset_index(drop=True)
log("")

=== 3b. INCONSISTENT RECORDS ===
Husband/Female or Wife/Male contradictions removed: 4



### 5. Outliers

In [9]:
# ---------- 4. Checking and handling outliers ----------
# Here I focus on the numerical columns to check if there are any unusual values.
# I use two different methods (IQR and Z-score) to identify values that are
# very far from the rest of the data.

num_cols = ["age", "fnlwgt", "education_num", "capital_gain",
            "capital_loss", "hours_per_week"]


# This function uses the IQR method to find values that are unusually low or high.
# It checks how far values are from the middle 50% of the data.
def iqr_outliers(s: pd.Series) -> int:
    """Number of values outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR] (Tukey's rule)."""
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())


# This second function uses Z-scores to find values that are more than
# 3 standard deviations away from the average.
def z_outliers(s: pd.Series, thr: float = 3.0) -> int:
    """Number of values with |z-score| > thr."""
    z = (s - s.mean()) / s.std()
    return int((z.abs() > thr).sum())


# I put the results from both methods into one table.
# I also include the minimum and maximum values to get a better idea
# of the ranges in each numerical column.
out_tbl = pd.DataFrame({
    "IQR_outliers": {c: iqr_outliers(df[c]) for c in num_cols},
    "Z>3_outliers": {c: z_outliers(df[c]) for c in num_cols},
    "min": df[num_cols].min(),
    "max": df[num_cols].max(),
})
log("=== 4. OUTLIERS (before treatment) ===")
log(out_tbl.to_string())


# ---------- Handling capital_gain ----------
# I noticed that capital_gain contains the value 99999, which is a special
# value used by the original census data to represent very large amounts.
# I flag these records and replace the value with the highest genuine amount
# so it doesn't have too much influence on my analysis.

sentinel = df["capital_gain"] == 99999
log(f"capital_gain == 99999 (top-code sentinel): {sentinel.sum()} rows")

df["capital_gain_topcoded"] = sentinel.astype(int)
true_max = df.loc[~sentinel, "capital_gain"].max()
df.loc[sentinel, "capital_gain"] = true_max

log(f"  -> capped at largest real value {true_max}")


# ---------- Creating new capital features ----------
# Most people have zero capital_gain or capital_loss, which makes these columns
# very uneven. Instead of removing these values, I create some extra columns
# to make the information easier to use during the analysis.
#
# I calculate the net capital amount and use log1p to reduce the effect
# of extremely large values while keeping the original information useful.

df["capital_net"] = df["capital_gain"] - df["capital_loss"]
df["log_capital_gain"] = np.log1p(df["capital_gain"])
df["log_capital_loss"] = np.log1p(df["capital_loss"])

log(f"Share of zero capital_gain: {(df['capital_gain'] == 0).mean():.3f}")


# ---------- Handling working hours ----------
# Some people have unusually low or high working hours.
# I use the 1st and 99th percentiles to limit these extreme values.
# This keeps the records in the dataset without allowing extreme values
# to have too much influence on the results.

lo, hi = df["hours_per_week"].quantile([0.01, 0.99])
n_w = ((df["hours_per_week"] < lo) | (df["hours_per_week"] > hi)).sum()

df["hours_per_week"] = df["hours_per_week"].clip(lo, hi)

log(f"hours_per_week winsorised to [{lo:.0f}, {hi:.0f}] -> {n_w} values changed")


# ---------- Keeping other numerical columns ----------
# I decided to keep age as it is because its values are reasonable for this dataset.
# I also keep fnlwgt, but exclude it from the main analysis because it represents
# a survey sampling weight rather than a personal characteristic.

log("age kept (17-90 plausible); fnlwgt kept but excluded from analysis (sampling weight)")
log("")

=== 4. OUTLIERS (before treatment) ===
                IQR_outliers  Z>3_outliers    min      max
age                      215           185     17       90
fnlwgt                  1453           506  12285  1490400
education_num           1787           326      1       16
capital_gain            4035           331      0    99999
capital_loss            2282          2216      0     4356
hours_per_week         13485           681      1       99
capital_gain == 99999 (top-code sentinel): 244 rows
  -> capped at largest real value 41310
Share of zero capital_gain: 0.917
hours_per_week winsorised to [8, 80] -> 773 values changed
age kept (17-90 plausible); fnlwgt kept but excluded from analysis (sampling weight)



### 6. Helper features for EDA

In [10]:
# ---------- Creating new features ----------
# I create a new column called income_gt50k to make it easier to compare
# people earning above and below 50K. I convert the income categories into
# numbers (1 = above 50K and 0 = 50K or below), which is useful for analysis.

df["income_gt50k"] = (df["income"] == ">50K").astype(int)


# I also group people's ages into different ranges instead of looking
# at every individual age. This makes it easier to compare income patterns
# between different age groups.
df["age_group"] = pd.cut(df["age"], bins=[16, 25, 35, 45, 55, 65, 91],
                         labels=["17-25", "26-35", "36-45", "46-55", "56-65", "66+"])


# ---------- Checking education information ----------
# I wanted to check if education and education_num contain the same information.
# If each education category always has the same numerical value, there is
# no need to treat them as completely separate pieces of information.
check = df.groupby("education")["education_num"].nunique().max()
log(f"education -> education_num is 1:1 mapping: {check == 1}")


# ---------- Converting columns into categories ----------
# Most of these columns contain repeated text values, such as occupation,
# workclass and marital status. I convert them to category types to make
# the dataset more efficient and easier to use when creating graphs.
for c in ["workclass", "marital_status", "occupation", "relationship",
          "race", "sex", "native_country", "income"]:
    df[c] = df[c].astype("category")

education -> education_num is 1:1 mapping: True


### 7. Keep the clean data set

In [11]:
# ---------- 5. Final check of the cleaned dataset ----------
# After completing the cleaning steps, I do one final check to see how
# the dataset looks. I check its size and the distribution of income
# to make sure everything is ready for the next stages of the project.

log("=== 5. CLEAN DATA ===")

# Checks the final number of rows and columns against the assignment requirements.
log(f"Final shape: {df.shape}  (requirement: >=3000 rows and >=7 columns)")

# Shows the percentage of people earning above and below 50K.
# This helps me understand how balanced the two income groups are.
log(f"Class balance: {df['income'].value_counts(normalize=True).round(3).to_dict()}")

# I create a separate copy of the cleaned dataset so I can use it
# in Parts 2 and 3 without having to repeat the cleaning process.
df_clean = df.copy()

=== 5. CLEAN DATA ===
Final shape: (48786, 21)  (requirement: >=3000 rows and >=7 columns)
Class balance: {'<=50K': 0.761, '>50K': 0.239}
